In [1]:
# import muon
import numpy as np
import mudata as md
import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import pandas as pd

# clustering
from sklearn.cluster import KMeans

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


/opt/miniconda3/envs/mudata/lib/python3.10/site-packages/louvain/__init__.py:54: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import get_distribution, DistributionNotFound


In [ ]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
import torch
import scipy.sparse as sp
import os
os.environ["MKL_VERBOSE"] = "0"
os.environ["MKL_DISABLE_FAST_MM"] = "1"
import warnings
warnings.filterwarnings("ignore", message="Intel MKL")
warnings.filterwarnings("ignore", category=FutureWarning)


def compute_lisi(emb, labels, perplexity=30):
    n_neighbors = int(3 * perplexity)
    nn = NearestNeighbors(n_neighbors=n_neighbors + 1).fit(emb)
    distances, indices = nn.kneighbors(emb)
    indices = indices[:, 1:]  # drop self

    labels = np.array(labels)
    out = []

    for i in range(emb.shape[0]):
        neigh = labels[indices[i]]
        p = pd.value_counts(neigh) / len(neigh)
        entropy = -(p * np.log(p)).sum()
        out.append(np.exp(entropy))

    return np.array(out)


def compute_morani(choice, rna_adata):
    graph_choice = "5_conn" if choice == "5" else choice
    graph_path = f"/Users/momo/Documents/cs536r/spatial_totalvi/data/graph/graph_k{graph_choice}.pt"

    graph_data = torch.load(graph_path)
    edge_index = graph_data['edge_index'].cpu().numpy()   # (2, E)
    edge_weight = graph_data['edge_weight'].cpu().numpy() # (E,)
    n_cells = rna_adata.n_obs

    spatial_graph_sparse = sp.coo_matrix(
        (edge_weight, (edge_index[0], edge_index[1])),
        shape=(n_cells, n_cells)
    )

    # Replace connectivities with spatial graph
    rna_adata.obsp['connectivities'] = spatial_graph_sparse.tocsr()

    rna_moran = sc.metrics.morans_i(
        rna_adata,
        obsm="X_spatial_totalVI"   # vals = latent dims
        # use_graph=None -> uses 'connectivities' we just set
    )
    return rna_moran


# ---------- Main loop ----------
choices = ["5", "20_conn", "100_conn"]
save_dir = "/Users/momo/Documents/cs536r/spatial_totalvi/benchmark/results"
os.makedirs(save_dir, exist_ok=True)

mdata = md.read_h5mu("../../data/tonsil/tonsil_pp_svg2.h5mu")
rna_adata_base = mdata.mod["RNA"]


metrics = []  

for i, choice in enumerate(choices):
    print(f"Processing model {choice} ...")

    rna_adata = rna_adata_base.copy()

    embeddings = pd.read_table(
        f"../../data/latents/totalvi_k{choice}_1layer_SGC_k1_latents.tsv",
        header=None, sep="\t"
    ).to_numpy()

    umap_rerun = pd.read_table(
        f"../../data/latents/totalvi_k{choice}_1layer_SGC_k1_umaplatents.tsv",
        header=None, sep="\t"
    ).to_numpy()

    rna_adata.obsm["X_spatial_totalVI"] = embeddings
    rna_adata.obsm["X_umap"] = umap_rerun

    sc.pp.neighbors(rna_adata, use_rep="X_spatial_totalVI", n_neighbors=30)
    sc.tl.umap(rna_adata)

    # ---------- Moran's I ----------
    rna_moran = compute_morani(choice, rna_adata)
    mean_moran = float(rna_moran.mean())

    # ---------- Clustering ----------
    kmeans = KMeans(n_clusters=5, random_state=0).fit(embeddings)
    rna_adata.obs['kmeans5'] = kmeans.labels_.astype(str)

    sc.tl.leiden(rna_adata, key_added="leiden_eval", resolution=0.7, random_state=0)
    sc.tl.louvain(rna_adata, key_added="louvain_eval", resolution=0.7, random_state=0)

    # ---------- cLISI ----------
    cLISI_kmean   = compute_lisi(embeddings, rna_adata.obs["kmeans5"])
    cLISI_louvain = compute_lisi(embeddings, rna_adata.obs["louvain_eval"])
    cLISI_leiden  = compute_lisi(embeddings, rna_adata.obs["leiden_eval"])

    mean_cLISI_kmeans   = float(cLISI_kmean.mean())
    mean_cLISI_louvain  = float(cLISI_louvain.mean())
    mean_cLISI_leiden   = float(cLISI_leiden.mean())

    # ---------- DBI ----------
    dbi_kmeans  = float(davies_bouldin_score(embeddings, kmeans.labels_))
    dbi_louvain = float(davies_bouldin_score(embeddings, rna_adata.obs["louvain_eval"].astype(int).values))
    dbi_leiden  = float(davies_bouldin_score(embeddings, rna_adata.obs["leiden_eval"].astype(int).values))


    # ---------- Collect metrics ----------
    metrics.append({
        "model": choice,
        "mean_moran": mean_moran,
        "mean_cLISI_kmeans5": mean_cLISI_kmeans,
        "mean_cLISI_louvain": mean_cLISI_louvain,
        "mean_cLISI_leiden": mean_cLISI_leiden,
        "dbi_kmeans": dbi_kmeans,
        "dbi_louvain": dbi_louvain,
        "dbi_leiden": dbi_leiden,
    })


metrics_df = pd.DataFrame(metrics)

metrics_df.to_csv(
    os.path.join(save_dir, "graph_metrics.tsv"),
    sep="\t",
    index=False
)
metrics_df

Processing model 5 ...
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Processing model 20_conn ...
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Adva

,model,mean_moran,mean_cLISI_kmeans5,mean_cLISI_louvain,mean_cLISI_leiden,dbi_kmeans,dbi_louvain,dbi_leiden
0,5,0.535132,1.768712,9.619024,10.644405,1.630314,6.451023,7.542200
1,20_conn,0.362846,1.496281,8.219830,8.350823,1.314619,11.095217,10.035390
2,100_conn,0.070919,2.366198,5.855993,5.258607,2.517038,11.070581,11.935858
